# Modular model building


In [50]:
import distopf as opf
import pyomo.environ as pyo
from distopf.pyomo_models.lindist import create_lindist_model
from distopf.pyomo_models import constraints
from distopf.pyomo_models.results import PyoResult
from distopf.api import create_case

First create the model

In [51]:
"""
Load the case into the Case object which contains all of the input data.
Case provides access to the following parameters:
    branch_data: pd.DataFrame,
    bus_data: pd.DataFrame,
    gen_data: Optional[pd.DataFrame] = None,
    cap_data: Optional[pd.DataFrame] = None,
    reg_data: Optional[pd.DataFrame] = None,
    bat_data: Optional[pd.DataFrame] = None,
    schedules: Optional[pd.DataFrame] = None,
    start_step: int = 0,
    n_steps: int = 1,
    delta_t: float = 1,  # hours per step
"""
case = create_case(
    data_path=opf.CASES_DIR / "csv" / "ieee123_30der", start_step=0, n_steps=24
)
# Make any modifications to the case dataframes using Pandas APIs
# Here we ensure that active and reactive power from generators are control variables.
case.gen_data.control_variable = "PQ"
# Create the pyomo ConcreteModel containint all of the
# necessary parameters, sets, and variables for the LinDist model.
model = create_lindist_model(case)

The `model` does not have any constraints yet so we need to add them.

In [52]:
# Power Flow Constraints
constraints.add_p_flow_constraints(model)
constraints.add_q_flow_constraints(model)
# Node Voltage Constraints
constraints.add_voltage_limits(model)
constraints.add_voltage_drop_constraints(model)
constraints.add_swing_bus_constraints(model)
# Loads, Capacitors and Regulators
constraints.add_cvr_load_constraints(model)
constraints.add_capacitor_constraints(model)
constraints.add_regulator_constraints(model)
# Generators
constraints.add_generator_limits(model)
constraints.add_generator_constant_p_constraints_q_control(model)
constraints.add_generator_constant_q_constraints_p_control(model)
#  - Choose the quadratic circular constraint or the linear octagonal constraint.
# constraints.add_circular_generator_constraints_pq_control(model)
constraints.add_octagonal_inverter_constraints_pq_control(model)
# Battery models
constraints.add_battery_constant_q_constraints_p_control(model)
constraints.add_battery_energy_constraints(model)
constraints.add_battery_net_p_bat_equal_phase_constraints(model)
constraints.add_battery_power_limits(model)
constraints.add_battery_soc_limits(model)

Let us add the following objective function:

$$\min \sum_{t \in \mathcal{T}} \sum_{p \in \phi_j, j:i \rightarrow j} \left(\left(P^{pp}_{ij}(t)\right)^2+\left(Q^{pp}_{ij}(t)\right)^2\right)r^{pp}_{ij}$$

Where:

- $P^{pp}_{ij}(t)$ is the active power flow from bus $i$ to bus $j$ on phase $p$ at time $t$ (`model.p_flow[i, j, p, t]`)
- $Q^{pp}_{ij}(t)$ is the reactive power flow from bus $i$ to bus $j$ on phase $p$ at time $t$ (`model.q_flow[i, j, p, t]`)
- $r^{pp}_{ij}$ is the resistance of branch from bus $i$ to bus $j$ on phase $p$ (`model.r[i, j, p+p]`)
- $p \in \phi_j, j:i \rightarrow j$ represents all phases $p$ and branches from bus $i$ to bus $j$ (`model.branch_phase_set`)
- $\mathcal{T}$ is the set of time steps (`model.time_set`)

In [53]:
model.objective = pyo.Objective(
    rule=pyo.quicksum(
        (model.p_flow[i, j, p, t] ** 2 + model.q_flow[i, j, p, t] ** 2)
        * model.r[i, j, p + p]
        for i, j, p in model.branch_phase_set
        for t in model.time_set
    ),
    sense=pyo.minimize,
)

Now the model is ready to solve.

In [54]:
opt = pyo.SolverFactory("ipopt")
results = opt.solve(model)
print(results.solver.status)
# Extract result dataframes from model
sol = PyoResult(model, results)
t_plot = None  # Time step for plots to show.

KeyboardInterrupt: 

In [ ]:
opf.plot_batteries(sol.p_bat, sol.soc)

In [ ]:
opf.plot_voltages(sol.voltages, t=t_plot)

In [ ]:
opf.plot_pq(sol.p_flow, sol.q_flow, t=t_plot)

In [ ]:
opf.plot_polar(sol.p_flow, sol.q_flow, t=t_plot)

In [ ]:
opf.plot_pq(sol.p_gen, sol.q_gen, t=t_plot)

In [ ]:
opf.plot_polar(sol.p_gen, sol.q_gen, t=t_plot)

In [ ]:
opf.plot_network(
    case,
    v=sol.voltages,
    p_flow=sol.p_flow,
    q_flow=sol.q_flow,
    p_gen=sol.p_gen,
    q_gen=sol.q_gen,
    show_reactive_power=True,
    t=t_plot,
)

In [ ]:
# plot_gens(res.p_bat, res.q_bat).show(renderer="browser")
